# 🧴 올리브영 스킨케어 EDA
> 가성비 스킨케어 추천 시스템 프로젝트 - 탐색적 데이터 분석

**데이터**: MongoDB Atlas `oliveyoung_db` > `skincare`, `skincare_review`

**분석 목표**:
- 평점 분포 및 변별력 확인
- 가격-인기(리뷰수) 관계 분석
- 가격-만족도 관계 분석
- 할인율-만족도 관계 분석

## 0. 환경 설정

In [ ]:
!pip install pymongo

In [ ]:
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 한글 폰트 설치
import os
import matplotlib.font_manager as fm

os.system('apt-get install -y fonts-nanum')
os.system('fc-cache -fv')
fm.fontManager.__init__()

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

print("환경 설정 완료")

## 1. MongoDB 연결 및 데이터 로드

In [ ]:
# Colab IP 확인 (Atlas Network Access 등록용)
!curl ifconfig.me

In [ ]:
# MongoDB 연결
# ⚠️ 실행 전: MONGODB_URI에 본인 연결 주소 입력
MONGODB_URI = "your_mongodb_connection_string_here"

client = pymongo.MongoClient(MONGODB_URI)
db = client["oliveyoung_db"]

# 연결 확인
print("컬렉션 목록:", db.list_collection_names())

In [ ]:
# DataFrame 변환
skincare_df = pd.DataFrame(list(db["skincare"].find()))
review_df = pd.DataFrame(list(db["skincare_review"].find()))

print("상품 데이터:", skincare_df.shape)   # (100, 21)
print("리뷰 데이터:", review_df.shape)      # (60165, 11)

## 2. 기본 현황 파악

In [ ]:
# 컬럼 목록 확인
print("[상품 컬럼]")
print(skincare_df.columns.tolist())

print("\n[리뷰 컬럼]")
print(review_df.columns.tolist())

In [ ]:
# 데이터 타입 및 non-null 개수
skincare_df.info()

## 3. 결측값 분석

In [ ]:
print("=" * 40)
print("[상품 데이터 결측값]")
print(skincare_df.isnull().sum())
print("\n결측값 비율 (%)")
print((skincare_df.isnull().mean() * 100).round(1))

print("=" * 40)
print("[리뷰 데이터 결측값]")
print(review_df.isnull().sum())
print("\n결측값 비율 (%)")
print((review_df.isnull().mean() * 100).round(1))

### 결측값 처리
- `original_price` → `sale_price`로 대체
- `skin_type`, `skin_tone` → 리뷰 텍스트 키워드 추출로 보완 후 미확인은 `정보없음` 처리

In [ ]:
# original_price 결측 → sale_price로 대체
skincare_df['original_price'] = skincare_df['original_price'].fillna(skincare_df['sale_price'])

# skin_type: 리뷰 텍스트에서 키워드 추출
def extract_skin_type(text):
    if not isinstance(text, str):
        return None
    if '지성' in text:
        return 'A01'
    elif '건성' in text:
        return 'A02'
    elif '복합' in text:
        return 'A03'
    elif '민감' in text:
        return 'A04'
    elif '중성' in text:
        return 'A05'
    return None

mask = review_df['skin_type'].isna()
review_df.loc[mask, 'skin_type'] = review_df.loc[mask, 'review_text'].apply(extract_skin_type)

# 여전히 결측인 것 → 정보없음
review_df['skin_type'] = review_df['skin_type'].fillna('정보없음')
review_df['skin_tone'] = review_df['skin_tone'].fillna('정보없음')

# 결과 확인
filled = (review_df['skin_type'] != '정보없음').sum()
print(f"skin_type 채워진 데이터: {filled}개 ({filled/len(review_df)*100:.1f}%)")
print(f"skin_type 정보없음: {(review_df['skin_type'] == '정보없음').sum()}개")

## 4. 기술 통계

In [ ]:
skincare_df[['original_price', 'sale_price', 'discount_rate',
             'review_count_total', 'average_rating', 'price_per_ml']].describe().round(2)

## 5. 핵심 분석 및 시각화

### 5-1. 인사이트 1: 평점 분포 — 변별력 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 상품 평균 평점 분포
axes[0].hist(skincare_df['average_rating'].dropna(), bins=20,
             color='lightgreen', edgecolor='black')
axes[0].axvline(skincare_df['average_rating'].mean(), color='red',
                linestyle='--', label=f"평균: {skincare_df['average_rating'].mean():.2f}")
axes[0].set_title('상품 평균 평점 분포')
axes[0].set_xlabel('평점')
axes[0].set_ylabel('빈도')
axes[0].legend()

# 리뷰 점수 분포
axes[1].hist(review_df['review_score'].dropna(), bins=5,
             color='plum', edgecolor='black')
axes[1].set_title('리뷰 점수 분포')
axes[1].set_xlabel('점수')
axes[1].set_ylabel('빈도')

plt.suptitle('[인사이트 1] 평점 분포 분석', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 수치 근거
high_rating = (skincare_df['average_rating'] >= 4.5).sum()
print(f"\n4.5점 이상 제품: {high_rating}개 ({high_rating}%)")
print("→ 평점만으로는 제품 변별이 어려움. 리뷰 텍스트 감성 분석 필요")

### 5-2. 인사이트 2: 가격과 인기(리뷰수) 관계

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 산점도
axes[0].scatter(skincare_df['sale_price'], skincare_df['review_count_total'],
                alpha=0.6, color='skyblue', edgecolor='gray')
axes[0].set_title('가격 vs 리뷰수 산점도')
axes[0].set_xlabel('판매가 (원)')
axes[0].set_ylabel('리뷰 수')

# 가격대별 평균 리뷰수
skincare_df['price_tier'] = pd.cut(
    skincare_df['sale_price'],
    bins=[0, 10000, 30000, 60000, 999999],
    labels=['저가(~1만)', '중가(1~3만)', '고가(3~6만)', '프리미엄(6만+)']
)
tier_review = skincare_df.groupby('price_tier', observed=True)['review_count_total'].mean()
tier_review.plot(kind='bar', ax=axes[1], color='skyblue', edgecolor='gray')
axes[1].set_title('가격대별 평균 리뷰수')
axes[1].set_xlabel('가격대')
axes[1].set_ylabel('평균 리뷰수')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('[인사이트 2] 가격과 인기(리뷰수) 관계', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("가격대별 평균 리뷰수:")
print(tier_review.round(0))
print("→ 중저가 제품에 리뷰 집중. 고가 제품은 콜드스타트 문제 가능")

### 5-3. 인사이트 3: 가격과 만족도 관계

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 산점도
axes[0].scatter(skincare_df['sale_price'], skincare_df['average_rating'],
                alpha=0.6, color='salmon', edgecolor='gray')
axes[0].set_title('가격 vs 평점 산점도')
axes[0].set_xlabel('판매가 (원)')
axes[0].set_ylabel('평점')

# 가격대별 평균 평점
tier_rating = skincare_df.groupby('price_tier', observed=True)['average_rating'].mean()
tier_rating.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='gray')
axes[1].set_title('가격대별 평균 평점')
axes[1].set_xlabel('가격대')
axes[1].set_ylabel('평균 평점')
axes[1].set_ylim(0, 5)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('[인사이트 3] 가격과 만족도 관계', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr = skincare_df['sale_price'].corr(skincare_df['average_rating'])
print(f"가격-평점 상관계수: {corr:.3f}")
print("→ 상관관계 거의 없음. 고가 = 고만족이 아님. 중저가 제품도 추천 유효")

### 5-4. 인사이트 4: 할인율과 만족도 관계

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 산점도
axes[0].scatter(skincare_df['discount_rate'], skincare_df['average_rating'],
                alpha=0.6, color='plum', edgecolor='gray')
axes[0].set_title('할인율 vs 평점 산점도')
axes[0].set_xlabel('할인율 (%)')
axes[0].set_ylabel('평점')

# 상관관계 히트맵
corr_matrix = skincare_df[['sale_price', 'discount_rate',
                             'review_count_total', 'average_rating',
                             'price_per_ml']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1])
axes[1].set_title('변수 간 상관관계 히트맵')

plt.suptitle('[인사이트 4] 할인율과 만족도 관계', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr = skincare_df['discount_rate'].corr(skincare_df['average_rating'])
print(f"할인율-평점 상관계수: {corr:.3f}")
print("→ 프로모션이 실제 품질과 무관. 할인율을 핵심 추천 피처로 쓰는 건 신중해야 함")

## 6. 리뷰 텍스트 기초 분석

In [ ]:
# 리뷰 텍스트 길이 분석
review_df['text_length'] = review_df['review_text'].str.len()

print("[리뷰 텍스트 길이 통계]")
print(review_df['text_length'].describe().round(1))

print("\n[점수별 평균 리뷰 길이]")
score_length = review_df.groupby('review_score')['text_length'].mean().round(1)
print(score_length)
print("\n→ 낮은 점수일수록 리뷰가 길어지는 경향 → 부정 리뷰 텍스트 분석이 유효")

## 7. 판매량 Proxy 산출 (베이지안 보정)

In [ ]:
# 베이지안 평균으로 리뷰 적은 고가 제품 보정
# C: 평균 리뷰수 (신뢰도 기준값)
# m: 전체 평균 평점

C = skincare_df['review_count_total'].mean()
m = skincare_df['average_rating'].mean()

skincare_df['bayesian_rating'] = (
    (skincare_df['review_count_total'] * skincare_df['average_rating'] + C * m)
    / (skincare_df['review_count_total'] + C)
)

# 로그 스케일 판매량 proxy
skincare_df['sales_proxy_log'] = np.log1p(skincare_df['review_count_total'])

print("[베이지안 보정 전후 비교 - 리뷰 적은 제품 상위 5개]")
low_review = skincare_df.nsmallest(5, 'review_count_total')
print(low_review[['product_name', 'review_count_total',
                   'average_rating', 'bayesian_rating']].to_string())
print("\n→ 리뷰 적은 제품의 평점이 전체 평균 방향으로 조정됨")

## 8. EDA 요약 및 모델링 연결 포인트

| EDA 결과 | 모델링 적용 |
|---|---|
| 평점 변별력 없음 | 감성 점수를 Quality Score에 포함 |
| 고가 제품 리뷰 부족 | 베이지안 보정으로 판매량 proxy 산출 |
| 가격-평점 무관계 | 가격 가중치(wp) 독립적으로 설정 |
| skin_type 결측 61% | 텍스트 키워드 보완 후 조건부 리랭킹 |

### Value Score 계산 구조
```
Value Score = wp × Price Score + wq × Quality Score

- wp, wq: 회귀분석(판매량 proxy ~ 각 점수) 또는 유저 입력 기반 결정
- Quality Score: 감성점수 + 품질점수 + 사회적점수 (NLP팀)
- 피부타입 기반 리랭킹: 유저 입력 시에만 조건부 적용
```